## Amazon AgentCore Bedrock Code Interpreter를 사용한 에이전트 기반 코드 실행 - 튜토리얼(LangChain)
이 튜토리얼에서는 Python 코드 실행을 통해 답변을 검증하는 AI 에이전트를 만드는 방법을 알아봅니다. Amazon Bedrock AgentCore Code Interpreter를 사용하여 LLM이 생성한 코드를 실행합니다.

AgentCore Bedrock Code Interpreter를 사용하여 다음 작업을 수행합니다.
1. 샌드박스 환경 설정
2. 사용자 질의를 바탕으로 코드를 생성하는 LangChain 기반 에이전트 구성
3. Code Interpreter를 사용하여 샌드박스 환경에서 코드 실행
4. 사용자에게 결과 표시

## 사전 요구 사항
- Bedrock AgentCore Code Interpreter에 액세스할 수 있는 AWS 계정
- Code Interpreter 리소스를 생성하고 관리하는 데 필요한 IAM 권한
- 필수 Python 패키지 설치(boto3, bedrock-agentcore 및 langchain 포함)
- Amazon Bedrock의 모델을 호출할 수 있는 권한이 있는 IAM 역할
 - 미국 오리건(us-west-2) 리전의 Claude 3.5 Sonnet 모델 액세스 권한

## IAM 실행 역할에 다음 IAM 정책을 연결해야 합니다



~~~ {
"Version": "2012-10-17",
"Statement": [
    {
        "Effect": "Allow",
        "Action": [
            "bedrock-agentcore:CreateCodeInterpreter",
            "bedrock-agentcore:StartCodeInterpreterSession",
            "bedrock-agentcore:InvokeCodeInterpreter",
            "bedrock-agentcore:StopCodeInterpreterSession",
            "bedrock-agentcore:DeleteCodeInterpreter",
            "bedrock-agentcore:ListCodeInterpreters",
            "bedrock-agentcore:GetCodeInterpreter"
        ],
        "Resource": "*"
    },
    {
        "Effect": "Allow",
        "Action": [
            "logs:CreateLogGroup",
            "logs:CreateLogStream",
            "logs:PutLogEvents"
        ],
        "Resource": "arn:aws:logs:*:*:log-group:/aws/bedrock-agentcore/code-interpreter*"
    }
]
}

## 작동 방식

코드 실행 샌드박스는 Code Interpreter, 셸, 파일 시스템을 갖춘 격리 환경을 생성하여 에이전트가 사용자 질의를 안전하게 처리할 수 있도록 합니다. 대규모 언어 모델(LLM)이 도구 선택을 지원한 후 이 세션 내에서 코드가 실행되며, 결과는 종합을 위해 사용자 또는 에이전트에게 반환됩니다.

![로컬 아키텍처](code-interpreter.png)

## 1. 환경 설정

먼저 필요한 라이브러리를 가져오고 Code Interpreter 세션을 초기화합니다.

In [ ]:
!pip install --upgrade -r requirements.txt

In [17]:
from bedrock_agentcore.tools.code_interpreter_client import code_session
from langchain.agents import (
    AgentExecutor,
    create_tool_calling_agent,
    tool,
)
from langchain_aws import ChatBedrockConverse
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
import json

## 2. 시스템 프롬프트 정의
AI 어시스턴트의 동작과 기능을 정의합니다. 항상 코드 실행과 데이터 기반 추론을 통해 답변을 검증하도록 지시합니다.

In [8]:
SYSTEM_PROMPT = """You are a helpful AI assistant that validates all answers through code execution.

VALIDATION PRINCIPLES:
1. When making claims about code, algorithms, or calculations - write code to verify them
2. Use execute_python to test mathematical calculations, algorithms, and logic
3. Create test scripts to validate your understanding before giving answers
4. Always show your work with actual code execution
5. If uncertain, explicitly state limitations and validate what you can

APPROACH:
- If asked about a programming concept, implement it in code to demonstrate
- If asked for calculations, compute them programmatically AND show the code
- If implementing algorithms, include test cases to prove correctness
- Document your validation process for transparency
- The sandbox maintains state between executions, so you can refer to previous results

TOOL AVAILABLE:
- execute_python: Run Python code and see output

RESPONSE FORMAT: The execute_python tool returns a JSON response with:
- sessionId: The sandbox session ID
- id: Request ID
- isError: Boolean indicating if there was an error
- content: Array of content objects with type and text/data
- structuredContent: For code execution, includes stdout, stderr, exitCode, executionTime"""

## 3. 코드 실행 도구 정의
다음으로 코드 샌드박스에서 코드를 실행할 때 에이전트가 사용할 함수를 도구로 정의합니다. @tool 데코레이터를 사용하여 이 함수를 에이전트의 사용자 정의 도구로 지정합니다.

활성 Code Interpreter 세션에서는 지원되는 언어(Python, JavaScript)로 코드를 실행하고, 종속성 구성에 따른 라이브러리에 액세스하고, 시각화를 생성하고, 실행 간 상태를 유지할 수 있습니다.

In [11]:
@tool
def execute_python(code: str, description: str = "") -> str:
    """Execute Python code in the sandbox."""

    if description:
        code = f"# {description}\n{code}"

    print(f"\n Generated Code: {code}")

    with code_session("us-west-2") as code_client:
        response = code_client.invoke("executeCode", {"code": code, "language": "python", "clearContext": False})

    for event in response["stream"]:
        return json.dumps(event["result"])

## 4. 에이전트 구성
LangChain SDK를 사용하여 에이전트를 생성하고 구성합니다. 생성된 코드를 실행할 수 있도록 위에서 정의한 시스템 프롬프트와 도구를 에이전트에 제공합니다.

#### 4.1 언어 모델 초기화

In [6]:
llm = ChatBedrockConverse(model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0", region_name="us-west-2")

#### 4.2 프롬프트 템플릿 정의

In [9]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", SYSTEM_PROMPT),
        ("user", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)

#### 4.3 사용자 정의 도구 목록 생성

In [ ]:
tools = [execute_python]

### 4.4 에이전트 실행기 생성


In [ ]:
agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

## 5. 질의 정의
에이전트의 코드 실행 기능을 테스트할 샘플 질의를 정의합니다.

In [ ]:
query = "Can all the planets in the solar system fit between the earth and moon?"

## 6. 에이전트 호출 및 응답 처리
질의로 에이전트를 호출하고 응답을 처리합니다. 에이전트는 가설에서 추론을 시작하고, 가설을 코드로 변환한 후 Code Interpreter에서 실행하여 검증합니다.

In [ ]:
response = agent_executor.invoke({"input": query})

### 에이전트의 최종 응답...

In [ ]:
print(response["output"][0]["text"])